In [416]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime

# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

# 모든 컬럼 출력설정(선택)
pd.set_option('display.max_columns', None)

#데이터 불러오기 
df = pd.read_csv('total_data.csv',index_col=0)

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")


[행/컬럼 갯수]
행: 51279, 컬럼: 40



In [417]:
#주차 컬럼 날짜타입 변환 (범주->날짜형)
df['주차'] = pd.to_datetime(df['주차'], format='%Y%m%d')
df['주차'].info()

<class 'pandas.core.series.Series'>
Index: 51279 entries, 0 to 51278
Series name: 주차
Non-Null Count  Dtype         
--------------  -----         
51279 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 801.2 KB


In [418]:
# 결측치 확인 -> 없음 
df.isna().sum()

기간        0
주차        0
라인        0
성별        0
기획년도      0
시즌이월      0
상품년차      0
시즌        0
복종        0
소품종       0
CAT       0
총입고수량     0
총입고원가     0
총입고택가     0
총출고수량     0
총출고원가     0
총출고택가     0
판매액       0
판매수량      0
매출원가      0
판매택가      0
총판매액      0
총판매수량     0
총매출원가     0
총판매택가     0
물류재고수량    0
물류재고원가    0
물류재고택가    0
매장재고수량    0
매장재고원가    0
매장재고택가    0
재고수량      0
재고원가      0
재고택가      0
기간입고수량    0
기간입고원가    0
기간입고택가    0
기간출고수량    0
기간출고원가    0
기간출고택가    0
dtype: int64

In [419]:
# 중복값 확인 및 제거 -> 전체 중복 12개
df.duplicated().sum() 
df.drop_duplicates(inplace=True)
# df[df.duplicated(keep=False)].sort_values(by='주차')

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

[행/컬럼 갯수]
행: 51267, 컬럼: 40



# 범주형 컬럼 확인

In [420]:
#성별 소품종 클래스 확인 : 기타로 분류되는 클래스 2개 존재
#--> 최종 '기타' 로 오분류된 항목 52개 변환
display(df['성별'].value_counts())

sex_df = df[df['성별'].str.contains('기타')]
sex_df['소품종'].value_counts()

#봄 패딩 베스트만, '기타' 로 분류됨 -> 성별 구분 착오 예상 --> 봄패딩베스트 '1:남성' 값으로 변환 
con = (df['소품종']=='패딩베스트') & (df['시즌']=='봄')
df.loc[con,'성별'] = '1:남성'

df.loc[con,'성별'].value_counts()

성별
1:남성      47346
3:남녀공용     2510
2:여성       1322
4:기타         89
Name: count, dtype: int64

성별
1:남성    52
Name: count, dtype: int64

In [421]:
#시즌이월 컬럼 클래스 확인 : 이월제품 의미 파악 필요
##결론 : 판매예측/할인최적화 모델링시 '이월' 행 삭제 ( -12578 32%) ,년간 매출 집계시 유지
display(df['시즌이월'].value_counts())

#2024년도 기준 시즌/이월 여부 확인 
df_2024 = df[df['기획년도']==2024]
df_2024 = df_2024.drop(columns=['기간','상품년차'])

# # 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df_2024['카테고리'] = df_2024['시즌'] + "_" +  df_2024['복종'] + "_" + df_2024['소품종'] + "_" + df_2024['라인']+ "_" + df_2024['성별']

# #카테고리별 시즌/이월값이 둘다 있는거 소팅 -> '샘플 가을_니트 셔츠_라운드_ZB_1:남성'   확인
df_2024.groupby('카테고리')['시즌이월'].nunique()

# #샘플확인 : '가을_니트 셔츠_라운드_ZB_1:남성' -> 시즌별 마감 이후 이월로 변경 됨 
con = df_2024['카테고리'] == '가을_니트 셔츠_라운드_ZB_1:남성'
smpl = df_2024[con].sort_values(by='주차',ascending=True)
smpl[(smpl['주차'] >'2024-11-01') & (smpl['주차'] <='2024-12-30')]

# 시즌-> 이월 바뀌는 시점 함수화 : gpt
def find_transition_points(group):
    group = group.sort_values('주차')
    transition_rows = group[(group['시즌이월'].shift(1) == '01_시즌') & (group['시즌이월'] == '02_이월')]
    return transition_rows[['카테고리', '주차']]

transition_points = df_2024.groupby('카테고리', group_keys=False).apply(find_transition_points)

# 시즌 -> 이월로 바뀌는 주차만 출력
transition_points['주차'].unique()

시즌이월
01_시즌    38689
02_이월    12578
Name: count, dtype: int64

<DatetimeArray>
['2024-12-01 00:00:00', '2024-06-02 00:00:00', '2024-10-06 00:00:00']
Length: 3, dtype: datetime64[ns]

In [422]:
print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

df.head(3)

[행/컬럼 갯수]
행: 51267, 컬럼: 40



,기간,주차,라인,성별,기획년도,시즌이월,상품년차,시즌,복종,소품종,CAT,총입고수량,총입고원가,총입고택가,총출고수량,총출고원가,총출고택가,판매액,판매수량,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,물류재고수량,물류재고원가,물류재고택가,매장재고수량,매장재고원가,매장재고택가,재고수량,재고원가,재고택가,기간입고수량,기간입고원가,기간입고택가,기간출고수량,기간출고원가,기간출고택가
0,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,봄,우븐 셔츠,캐쥬얼셔츠,02_SHIRTS,2247,19893640,157065300,1009,8933103,70529100,-124850,0,0,0,84850,3,26560,209700,1238,10960537,86536200,1006,8906543,70319400,2244,19867080,156855600,0,0,0,53,469228,3704700
1,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,사계절,소품,양말,10_ACC/ETC,14000,11480000,46200000,10775,8835500,35557500,2033156,617,505940,2036100,10270838,3127,2564140,10319100,3225,2644500,10642500,7648,6271360,25238400,10873,8915860,35880900,0,0,0,454,372280,1498200
2,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,봄,니트 셔츠,라운드,01_KNIT,20076,178239296,1403312400,5940,52736683,415206000,18981424,401,3559559,28029900,18981424,401,3560170,28029900,14136,125502613,988106400,5539,49176513,387176100,19675,174679125,1375282500,0,0,0,5940,52713656,415206000


# 수치형 컬럼 & 집계 컬럼 점검
사용 컬럼 : '총입고수량','총입고원가','총입고택가','총출고수량','총출고원가','총출고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가'

In [423]:
# 총입고수량/입고원가/입고택가  -> 전처리 (-162행)
# 입고전 데이터 확인 및 행 삭제 : 162개 -> 입고되지 않은 상품은 출고 및 판매 불가, 예약판매 등 특수한 케이스 없다고 가정 

#1. 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df['카테고리'] = df['시즌'] + "_" +  df['복종'] + "_" + df['소품종'] + "_" + df['라인']
num_df = df[['카테고리','주차','총입고수량','총입고원가','총입고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가','시즌','복종','소품종','라인','시즌이월','기획년도']]

print((num_df[f'총입고수량'] == 0).sum())
filtered_df = num_df[(df['총입고수량'] > 0)]

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

162
[행/컬럼 갯수]
행: 51105, 컬럼: 22



In [424]:
# 재고수량/재고원가/재고택가 정합성 확인 -> 재고 관련 컬럼 삭제 
# 결론: 재고관련 집계 컬럼 삭제 후 입고-판매 기준 다시 집계 (입고,판매 데이터의 신뢰도가 더 높다고 봄, 실제 wms랑 비교 할 수 없으므로 가정)
filtered_df['재고잔량_check'] = (filtered_df['총입고수량'] - filtered_df['총판매수량'] == filtered_df['재고수량'])
filtered_df['재고원가_check'] = (filtered_df['총입고원가'] - filtered_df['총매출원가'] == filtered_df['재고원가'])
filtered_df['재고택가_check'] = (filtered_df['총입고택가'] - filtered_df['총판매택가'] == filtered_df['재고원가'])

# 입고 - 판매 = 재고 안맞는 행 : 27569행
filtered_df[filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False]
(filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False).sum() #27569행

# 재고관련 컬럼 삭제
filtered_df.drop(columns=['재고수량','재고원가','재고택가','재고잔량_check','재고원가_check','재고택가_check'],inplace=True)

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

[행/컬럼 갯수]
행: 51105, 컬럼: 19



# 최종 전처리 진행 

In [425]:
filtered_df.head(3)

,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,기획년도
0,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,-124850,0,0,84850,3,26560,209700,봄,우븐 셔츠,캐쥬얼셔츠,ZB,01_시즌,2021
1,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156,505940,2036100,10270838,3127,2564140,10319100,사계절,소품,양말,ZB,01_시즌,2021
2,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424,3559559,28029900,18981424,401,3560170,28029900,봄,니트 셔츠,라운드,ZB,01_시즌,2021


In [426]:
# 그룹화 및 집계 연산 적용 :최종 컬럼 21345행
group_cols = ['시즌','복종','소품종','라인','시즌이월','기획년도','카테고리','주차']
agg_dict = {
    '총입고수량': 'sum',
    '총입고원가': 'sum',
    '총입고택가': 'sum',

    '판매수량': 'sum',
    '판매액': 'sum',
    '매출원가': 'sum',
    '판매택가': 'sum'
}

filtered_df = filtered_df.groupby(group_cols).agg(agg_dict).reset_index()

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

[행/컬럼 갯수]
행: 21345, 컬럼: 15



- 파생변수 생성 

In [ ]:
# 1. 입고기준 원가/택가 -> 정수로 반올림
filtered_df['제품원가'] = (filtered_df['총입고원가'] / filtered_df['총입고수량']).round(0)
filtered_df['제품택가'] = (filtered_df['총입고택가'] / filtered_df['총입고수량']).round(0)

df['카테고리'].nunique()

252

In [ ]:
# 2. 카테고리 시즌별 평균 판매수량, 평균 실판가 집계 (아래 고려사항)
# 카테고리별 평균 판매수량
# 카테고리별 평균 실판가
# 시즌내 마감일자 기간 까지의 평균 값 적용 
# 정수까지 반올림

print('[최초 행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

#시즌 데이터만 (=마감일자까지의 평균 값 집계)
avg_df = filtered_df[filtered_df['시즌이월']=='01_시즌']

print('[마감일자까지 행/컬럼 갯수]')
print(f"행: {avg_df.shape[0]}, 컬럼: {avg_df.shape[1]}\n")

# 평균 판매수량 / 실판가 값 생성 (평균 실판가 = 판매액/판매수량 의 평균 )
# avg_df.describe(include='all')
avg_df['평균실판가'] = avg_df['판매액'] / avg_df['판매수량']

avg_df2 =avg_df.groupby(['카테고리','기획년도'])[['판매수량','평균실판가']].mean().reset_index()
avg_df2 = avg_df2.round(0) #평균값 
avg_df2



[최초 행/컬럼 갯수]
행: 21345, 컬럼: 17

[마감일자까지 행/컬럼 갯수]
행: 15439, 컬럼: 17



,카테고리,기획년도,판매수량,평균실판가
0,가을_니트 셔츠_라운드_ZB,2021,869.0,35654.0
1,가을_니트 셔츠_라운드_ZB,2022,202.0,40876.0
2,가을_니트 셔츠_라운드_ZB,2023,227.0,34667.0
3,가을_니트 셔츠_라운드_ZB,2024,197.0,19876.0
4,가을_니트 셔츠_라운드_ZD,2021,95.0,26250.0
...,...,...,...,...
609,여름_팬츠_팬츠(일반)_ZE,2024,933.0,27658.0
610,여름_팬츠_팬츠(일반)_ZF,2022,64.0,26010.0
611,여름_팬츠_팬츠(일반)_ZF,2023,29.0,51710.0
612,여름_팬츠_팬츠(일반)_ZF,2024,10.0,NaN


In [429]:
# 각 데이터프레임의 카테고리 개수 확인
filtered_count = filtered_df['카테고리'].nunique()
avg_count = avg_df2['카테고리'].nunique()

print(f"최초 카테고리 개수: {filtered_count}")
print(f"avg 카테고리 개수: {avg_count}")

#본 데이터에 평균실판가/평균판매수량 병합
merge_df = filtered_df.merge(avg_df2, on=['카테고리','기획년도'], how='inner') # **기획년도추가 

merge_df = merge_df.rename(columns={'판매수량_x' : '판매수량','판매수량_y' : '평균판매수량'})
merge_df.head(3)

#최종 merge_df


최초 카테고리 개수: 252
avg 카테고리 개수: 252


,시즌,복종,소품종,라인,시즌이월,기획년도,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균판매수량,평균실판가
0,가을,니트 셔츠,라운드,ZB,01_시즌,2021,가을_니트 셔츠_라운드_ZB,2021-07-11,10931,90858472,764076900,0,0,0,0,8312.0,69900.0,869.0,35654.0
1,가을,니트 셔츠,라운드,ZB,01_시즌,2021,가을_니트 셔츠_라운드_ZB,2021-07-18,10931,90858472,764076900,56,3131520,466312,3914400,8312.0,69900.0,869.0,35654.0
2,가을,니트 셔츠,라운드,ZB,01_시즌,2021,가을_니트 셔츠_라운드_ZB,2021-07-25,10931,90858472,764076900,-7,-506220,-59024,-489300,8312.0,69900.0,869.0,35654.0


In [430]:
# 음수 데이터 확인 -> 판매수량/판매액/매출원가/판매택가의 음수값 갯수가 다 다름
minus_con = merge_df.select_dtypes(include='number') <0
minus_con.sum()

기획년도        0
총입고수량       0
총입고원가       0
총입고택가       0
판매수량      737
판매액       636
매출원가      775
판매택가      746
제품원가        0
제품택가        0
평균판매수량      0
평균실판가     100
dtype: int64

In [431]:
# 1. 판매수량이 0이면서 판매액/매출원가/판매택가 중 하나라도 0이 아닌 경우 -> 126행 대치 완료 
#결론: 판매 없이 판매액/매출원가/판매택가 발생할수없다고 판단, 이상치로 간주하고 매출원가/판매택가 0으로 변경
mask = (merge_df['판매수량'] == 0) & (
        (merge_df['판매액'] != 0) | (merge_df['매출원가'] != 0) | (merge_df['판매택가'] != 0))

print(mask.sum())

merge_df.loc[mask, ['판매액', '매출원가', '판매택가']] = 0

merge_df.describe()

126


,기획년도,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균판매수량,평균실판가
count,21345.000000,21345,21345.000000,2.134500e+04,2.134500e+04,21345.000000,2.134500e+04,2.134500e+04,2.134500e+04,21345.000000,2.134500e+04,21345.000000,20589.0
mean,2022.560506,2023-02-28 20:48:20.238931968,19837.395737,2.475734e+08,2.096637e+09,375.862497,1.719693e+07,5.084203e+06,4.326007e+07,21084.848255,1.549335e+05,485.047646,NaN
min,2021.000000,2021-01-03 00:00:00,25.000000,4.053250e+05,4.997500e+06,-8003.000000,-1.911378e+08,-8.352608e+07,-8.841990e+08,166.000000,3.300000e+03,0.000000,-inf
25%,2022.000000,2022-03-20 00:00:00,3474.000000,5.953037e+07,4.883760e+08,3.000000,1.720570e+05,5.400000e+04,3.995000e+05,8686.000000,8.990000e+04,96.000000,29223.0
50%,2023.000000,2023-03-12 00:00:00,8944.000000,1.330766e+08,1.123812e+09,63.000000,3.191200e+06,1.004657e+06,8.022700e+06,14095.000000,1.090000e+05,230.000000,48112.0
75%,2024.000000,2024-01-14 00:00:00,20138.000000,3.011455e+08,2.484321e+09,334.000000,1.847634e+07,5.325624e+06,4.245750e+07,25777.000000,1.990000e+05,536.000000,98180.0
max,2024.000000,2024-12-29 00:00:00,739995.000000,3.281674e+09,3.883962e+10,38139.000000,6.367838e+08,3.591616e+08,3.854213e+09,266866.000000,1.199000e+06,12996.000000,inf
std,1.089211,NaN,43627.556512,3.279845e+08,3.103238e+09,1108.354726,3.552381e+07,1.181850e+07,1.122362e+08,23405.072196,1.229558e+05,916.954686,NaN


In [432]:
# 2.판매수량/ 매출원가 부호정합성 -> (제품원가 * 판매수량)
# 판매수량 >0 , 집계컬럼 <=0 (매출원가)
# 결론 :  집계 오류 판단 -> 입고원가 * 판매수량 으로 대치
con = (merge_df['판매수량'] > 0) & (merge_df['매출원가'] < 0)
display(con.sum())

merge_df.loc[con,'매출원가']= merge_df['제품원가'] * merge_df['판매수량'] 
merge_df[con]

# 판매수량 <0 , 집계컬럼 >=0 (매출원가)
# 결론 : 집계 오류 판단 -> 입고원가 * 판매수량 으로 대치
con = (merge_df['판매수량'] < 0) & (merge_df['매출원가'] >= 0)
display(con.sum())

merge_df.loc[con,'매출원가']= merge_df['제품원가']* merge_df['판매수량'] 
merge_df[con]

8

2

,시즌,복종,소품종,라인,시즌이월,기획년도,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균판매수량,평균실판가
791,가을,우븐 셔츠,캐쥬얼셔츠,ZB,02_이월,2024,가을_우븐 셔츠_캐쥬얼셔츠_ZB,2024-12-29,9036,78666703,713844000,-1,-2179253,-8706,-79000,8706.0,79000.0,245.0,47258.0
15814,여름,스웨터,T-에리,ZA,02_이월,2022,여름_스웨터_T-에리_ZA,2022-10-02,10633,77833939,1062236700,-1,-427530,-7320,-99900,7320.0,99900.0,191.0,32323.0


In [433]:
# 3.판매수량/ 판매액 부호정합성 -> (평균판매액 * 판매수량)
#판매수량 >0 , 집계컬럼 <0 (판매액)
#총 65행 
con1 = (merge_df['판매수량'] > 0) & (merge_df['판매액'] < 0)
display(con1.sum())

#판매수량 <0, 집계컬럼 >0 (판매액)
#총 133행 
con2 = (merge_df['판매수량'] < 0) & (merge_df['판매액'] > 0)
display(con2.sum())

#판매수량 !=0, 집계컬럼 ==0 (판매액) 
#총 88행 
con3 = (merge_df['판매수량'] != 0) & (merge_df['판매액'] == 0)
display(con3.sum())
merge_df[con3]

# 총 이상치 갯수
display((con1 | con2 | con3).sum())

merge_df.loc[(con1 | con2 | con3),'판매액'] = merge_df['판매수량'] * merge_df['평균실판가']
merge_df 

65

133

88

286

,시즌,복종,소품종,라인,시즌이월,기획년도,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균판매수량,평균실판가
0,가을,니트 셔츠,라운드,ZB,01_시즌,2021,가을_니트 셔츠_라운드_ZB,2021-07-11,10931,90858472,764076900,0,0.0,0,0,8312.0,69900.0,869.0,35654.0
1,가을,니트 셔츠,라운드,ZB,01_시즌,2021,가을_니트 셔츠_라운드_ZB,2021-07-18,10931,90858472,764076900,56,3131520.0,466312,3914400,8312.0,69900.0,869.0,35654.0
2,가을,니트 셔츠,라운드,ZB,01_시즌,2021,가을_니트 셔츠_라운드_ZB,2021-07-25,10931,90858472,764076900,-7,-506220.0,-59024,-489300,8312.0,69900.0,869.0,35654.0
3,가을,니트 셔츠,라운드,ZB,01_시즌,2021,가을_니트 셔츠_라운드_ZB,2021-08-01,35541,251528579,2484315900,-26,-1530810.0,-216112,-1817400,7077.0,69900.0,869.0,35654.0
4,가을,니트 셔츠,라운드,ZB,01_시즌,2021,가을_니트 셔츠_라운드_ZB,2021-08-08,42541,311300869,2973615900,425,23612190.0,3095254,29707500,7318.0,69900.0,869.0,35654.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21340,여름,팬츠,팬츠(일반),ZG,02_이월,2024,여름_팬츠_팬츠(일반)_ZG,2024-12-01,2600,50913796,335400000,0,0.0,0,0,19582.0,129000.0,58.0,52906.0
21341,여름,팬츠,팬츠(일반),ZG,02_이월,2024,여름_팬츠_팬츠(일반)_ZG,2024-12-08,2600,50913796,335400000,3,117000.0,58747,387000,19582.0,129000.0,58.0,52906.0
21342,여름,팬츠,팬츠(일반),ZG,02_이월,2024,여름_팬츠_팬츠(일반)_ZG,2024-12-15,2600,50913796,335400000,4,236000.0,78329,516000,19582.0,129000.0,58.0,52906.0
21343,여름,팬츠,팬츠(일반),ZG,02_이월,2024,여름_팬츠_팬츠(일반)_ZG,2024-12-22,2600,50913796,335400000,2,118000.0,39164,258000,19582.0,129000.0,58.0,52906.0


In [434]:
# 4.판매수량/ 판매택가 부호정합성 (제품택가 * 판매수량)
# 판매수량 >0 , 집계컬럼 <=0 (판매택가)
# 결론 : 3행 존재 집계 오류 판단 -> 입고택가 * 판매수량 으로 대치
con = (merge_df['판매수량'] > 0) & (merge_df['판매택가'] < 0)
display(con.sum())

merge_df.loc[con,'판매택가']= merge_df['제품택가'] * merge_df['판매수량'] 
merge_df[con]

# 판매수량 <0 , 집계컬럼 >=0 (판매택가)
# 결론 : 없음
con1 = (merge_df['판매수량'] < 0) & (merge_df['판매택가'] >= 0)
display(con1.sum())

merge_df[con]

3

0

,시즌,복종,소품종,라인,시즌이월,기획년도,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,제품원가,제품택가,평균판매수량,평균실판가
3623,겨울,자켓,싱글재킷,ZA,01_시즌,2024,겨울_자켓_싱글재킷_ZA,2024-12-29,10296,431502948,3379704000,16,NaN,670560,5252064,41910.0,328254.0,248.0,NaN
13769,여름,니트 셔츠,라운드,ZE,01_시즌,2023,여름_니트 셔츠_라운드_ZE,2023-09-17,106757,652698150,6509914300,17,191772.0,20282,1036643,6114.0,60979.0,2700.0,15559.0
20567,여름,팬츠,팬츠(일반),ZA,02_이월,2022,여름_팬츠_팬츠(일반)_ZA,2022-12-11,49379,704920915,6687887400,3,178623.0,122612,406320,14276.0,135440.0,920.0,59541.0


In [435]:
# 음수 데이터 확인 -> 판매수량/판매액/매출원가/판매택가의 음수값 갯수가 다 다름
minus_con = merge_df.select_dtypes(include='number') <0
minus_con.sum()

기획년도        0
총입고수량       0
총입고원가       0
총입고택가       0
판매수량      737
판매액       727
매출원가      737
판매택가      737
제품원가        0
제품택가        0
평균판매수량      0
평균실판가     100
dtype: int64

# 파생변수 추가
- 주차별 실판가/할인율 
- 누적판매수량 / 누적판매액 / 누적매출원가 /누적판매택가
- 누적판매율(수량) / roi 
** 정수단위 반올림 / roi 는 소수점 둘째까지 반올림

In [436]:
#제품실판가,할인율
merge_df['제품실판가'] = merge_df['판매액']/merge_df['판매수량'] #null 값 있음
merge_df['할인율(%)'] = (merge_df['판매택가'] - merge_df['판매액']) / merge_df['판매택가']*100 #null 값 있음

#누적판매데이터: cumsum()
merge_df = merge_df.sort_values(by=['카테고리','주차'],ascending=True)

merge_df['누적판매수량'] = merge_df.groupby(['기획년도','카테고리'])['판매수량'].cumsum()
merge_df['누적판매액'] = merge_df.groupby(['기획년도','카테고리'])['판매액'].cumsum()
merge_df['누적매출원가'] = merge_df.groupby(['기획년도','카테고리'])['매출원가'].cumsum()
merge_df['누적판매택가'] = merge_df.groupby(['기획년도','카테고리'])['판매택가'].cumsum()

#판매율 / roi
merge_df['누적판매율(%)'] = merge_df['누적판매수량']/merge_df['총입고수량']*100
merge_df['ROI'] = ((merge_df['누적판매액']/1.1 - merge_df['누적매출원가'])/merge_df['총입고원가']).round(2) #roi 소수점 2자리 까지

# 실수형 서식 변환 : 반올림0까지
round_cols = ['제품실판가', '할인율(%)','누적판매율(%)']
merge_df[round_cols] = merge_df[round_cols].round(0)

print('[행/컬럼 갯수]')
print(f"행: {merge_df.shape[0]}, 컬럼: {merge_df.shape[1]}\n")
final_df = merge_df 

[행/컬럼 갯수]
행: 21345, 컬럼: 27



In [ ]:
#csv 추출
# final_df.to_csv('기본전처리최종.csv')

# 최종 전처리 df : final_df

In [438]:
#이상치 판매수량 - 판매액 관련 고찰 

#추정 실판가(판매액/판매수량) 과 직전 실판가 비교  / 추정 판매수량 과 직전 판매수량 비교
# con1 = (filtered_df['판매수량'] > 0) & (filtered_df['판매액'] < 0)
# con2 = (filtered_df['판매수량'] < 0) & (filtered_df['판매액'] > 0)
# display((con1 | con2).sum())

# # 1. 데이터 정렬 (카테고리, 총입고수량, 주차 기준)
# filtered_df = filtered_df.sort_values(by=['카테고리','총입고수량','주차'])

# # 2. 직전 주차의 실판가 계산 & 직전 주차 판매수량
# filtered_df['직전주차_판매수량'] = filtered_df.groupby(['카테고리','총입고수량'])['판매수량'].shift(1)
# filtered_df['직전주차_판매액'] = filtered_df.groupby(['카테고리','총입고수량'])['판매액'].shift(1)

# filtered_df['직전주차_실판가'] = (filtered_df['직전주차_판매액'] / filtered_df['직전주차_판매수량']).round(0)
# filtered_df['추정_실판가'] = (filtered_df['판매액']/ filtered_df['판매수량']).round(0)
# # filtered_df['추정_실판가'] = filtered_df['추정_실판가'].fillna(0)  # NaN 값 0으로 대체
# filtered_df.loc[(con1|con2),['카테고리','주차','판매수량','판매액','직전주차_판매수량','추정_실판가','직전주차_실판가','매출원가','판매택가']]

# # 엑셀로 점검 
# a = filtered_df.loc[(con1|con2),['카테고리','주차','판매수량','판매액','직전주차_판매수량','추정_실판가','직전주차_실판가','매출원가','판매택가']]
# a.to_excel('check_sales.xlsx')

#판매액오류 가능성이 더 커보임 => 최종 : 해당 카테고리별 양수값의 평균 실판가로 대치 시키자 
## 1. 직전주차 실판가 로 대치시킨다면, 0이하이거나 null인 행 57개 추가로 어떻게 대치 시킬지 